In [ ]:
import numpy as np
from scipy.special import sph_harm_y, lpmn, ellip_harm

from generate_3d_mesh import spherical_mesh, visualize_mesh
from src.export_mesh import export_mesh

We need to define our coordinates as an array of points where each point is given as _[longitude, latitude, distance]_

In [ ]:
# Create a grid of longitude and latitude points
lon_steps, lat_steps = 15, 10
longitudes = np.linspace(-180, 180, lon_steps)
latitudes = np.linspace(-90, 90, lat_steps)

# Create a meshgrid for all combinations
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# Flatten the grid
lons = lon_grid.flatten()
lats = lat_grid.flatten()

In [ ]:
# Create distances with a simple pattern (a bumpy sphere)
base_radius = 10
distances = base_radius + 4 * np.sin(np.radians(lons)) * np.cos(np.radians(lats))

# Combine into points array
points = np.column_stack((lons, lats, distances))

print(f"Generated {len(points)} points")

In [ ]:
# Generate mesh
vertices, triangles = spherical_mesh(points)

print(f"Mesh created with {len(vertices)} vertices and {len(triangles)} triangles")

In [ ]:
# Visualize
fig = visualize_mesh(vertices, triangles, "Bumpy Sphere Example", renderer='notebook', return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)

## Spherical Harmonics

Spherical harmonics are a set of functions that form an orthogonal basis for functions defined on the surface of a sphere. They are particularly useful for representing complex 3D shapes and are commonly used in physics, computer graphics, and other fields.

We'll use `scipy.special.sph_harm_y` to create a 3D mesh based on spherical harmonics.

In [ ]:
# Create a finer grid for spherical harmonics
lon_steps_sh, lat_steps_sh = 15, 10
longitudes_sh = np.linspace(0, 360, lon_steps_sh)
latitudes_sh = np.linspace(-90, 90, lat_steps_sh)

# Create a meshgrid for all combinations
lon_grid_sh, lat_grid_sh = np.meshgrid(longitudes_sh, latitudes_sh)

# Flatten the grid
lons_sh = lon_grid_sh.flatten()
lats_sh = lat_grid_sh.flatten()

# Convert to radians for spherical harmonics calculation
phi = np.radians(lons_sh)  # Azimuthal angle (longitude)
theta = np.radians(90 - lats_sh)  # Polar angle (colatitude)

In [ ]:
# Define a single spherical harmonic
l = 3  # Degree
m = 2  # Order

# Calculate spherical harmonics
# Note: sph_harm_y takes (l, m, theta, phi) as arguments
sh_values = sph_harm_y(l, m, theta, phi)

# Take real part and normalize for visualization
sh_real = np.real(sh_values)
sh_normalized = (sh_real - np.min(sh_real)) / (np.max(sh_real) - np.min(sh_real))

# Create distances based on spherical harmonics
base_radius_sh = 10
amplitude = 3
distances_sh = base_radius_sh + amplitude * sh_normalized

# Combine into points array
points_sh = np.column_stack((lons_sh, lats_sh, distances_sh))

print(f"Generated {len(points_sh)} points using spherical harmonics (l={l}, m={m})")

In [ ]:
# Generate mesh from spherical harmonics
vertices_sh, triangles_sh = spherical_mesh(points_sh)

print(f"Mesh created with {len(vertices_sh)} vertices and {len(triangles_sh)} triangles")

In [ ]:
# Visualize spherical harmonics mesh
fig_sh = visualize_mesh(vertices_sh, triangles_sh, f"Spherical Harmonics (l={l}, m={m})", renderer='browser', return_fig=True)
fig_sh.update_layout(template='plotly_dark', showlegend=False)
fig_sh.show(renderer='notebook')

## Multiple Spherical Harmonics

We can create more complex shapes by combining multiple spherical harmonics with different parameters and weights.

In [ ]:
# Function to create a mesh from multiple spherical harmonics
def create_spherical_harmonics_mesh(harmonics, base_radius=10, amplitude=3, lon_steps=20, lat_steps=15):
    """
    Create a 3D mesh from multiple spherical harmonics.

    Args:
        harmonics (list): List of dictionaries, each with 'l', 'm', and 'weight' keys
        base_radius (float): Base radius of the sphere
        amplitude (float): Amplitude of the spherical harmonics
        lon_steps (int): Number of longitude steps
        lat_steps (int): Number of latitude steps

    Returns:
        tuple: (vertices, triangles, points) for the generated mesh
    """
    # Create a grid for spherical harmonics
    longitudes = np.linspace(0, 360, lon_steps)
    latitudes = np.linspace(-90, 90, lat_steps)

    # Create a meshgrid for all combinations
    lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

    # Flatten the grid
    lons = lon_grid.flatten()
    lats = lat_grid.flatten()

    # Convert to radians for spherical harmonics calculation
    phi = np.radians(lons)  # Azimuthal angle (longitude)
    theta = np.radians(90 - lats)  # Polar angle (colatitude)

    # Calculate combined spherical harmonics
    combined_sh = np.zeros_like(theta, dtype=complex)

    # Add each harmonic with its weight
    for harmonic in harmonics:
        l = harmonic['l']
        m = harmonic['m']
        weight = harmonic['weight']

        # Calculate spherical harmonic
        sh_values = sph_harm_y(l, m, theta, phi)

        # Add to combined value with weight
        combined_sh += weight * sh_values

    # Take real part and normalize for visualization
    combined_real = np.real(combined_sh)
    combined_normalized = (combined_real - np.min(combined_real)) / (np.max(combined_real) - np.min(combined_real))

    # Create distances based on combined spherical harmonics
    distances = base_radius + amplitude * combined_normalized

    # Combine into points array
    points = np.column_stack((lons, lats, distances))

    # Generate mesh
    vertices, triangles = spherical_mesh(points)

    return vertices, triangles, points

In [ ]:
# Example usage of the function with different parameters
example_harmonics = [
    {'l': 0, 'm': 0, 'weight': 0.7},
    {'l': 9, 'm': 8, 'weight': 0.2},
    {'l': 2, 'm': 0, 'weight': 0.1},
]

vertices_example, triangles_example, points_example = create_spherical_harmonics_mesh(
    example_harmonics, 
    base_radius=10, 
    amplitude=2,
    lon_steps=35,
    lat_steps=25,
)

# Generate a description of the harmonics used
example_desc = ", ".join([f"({h['l']},{h['m']}): {h['weight']}" for h in example_harmonics])
print(f"Generated mesh using harmonics: {example_desc}")
print(f"Mesh created with {len(vertices_example)} vertices and {len(triangles_example)} triangles")

In [ ]:
# Visualize the example mesh
fig_example = visualize_mesh(vertices_example, triangles_example, f"Custom Spherical Harmonics", return_fig=True)
fig_example.update_layout(template='plotly_dark', showlegend=False)
fig_example.show(renderer='notebook')


In [ ]:
# Export a mesh to a 3D file format that Blender can import.

# Args:
#     vertices (numpy.ndarray): Vertex coordinates array (n_vertices, 3)
#     triangles (numpy.ndarray): Triangle indices array (n_triangles, 3)
#     filename (str): Output filename (without extension)
#     export_format (str): Export format - 'obj', 'stl', or 'ply'
#
# Returns:
#     str: Path to the exported file

export_mesh(vertices_example, triangles_example, "output/spherical_harmonics_mesh", "stl")

## Multiple Ellipsoidal Harmonics

Similarly to above section, we can create more complex shapes by combining multiple ellipsoidal harmonics with different parameters and weights.

In [ ]:
def create_ellipsoidal_harmonics_mesh(harmonics, semi_axes=(10, 8, 6), amplitude=3, lon_steps=20, lat_steps=15):
    """
    Create a 3D mesh from multiple ellipsoidal harmonics using scipy.special.ellip_harm.

    Args:
        harmonics (list): List of dictionaries, each with 'h', 'k', 'n', and 'weight' keys
            where 'h' is the first kind (1 or 2), 'k' is the second kind (1 or 2),
            and 'n' is the degree
        semi_axes (tuple): Semi-axes (a, b, c) of the base ellipsoid where a > b > c > 0
        amplitude (float): Amplitude of the ellipsoidal harmonics
        lon_steps (int): Number of longitude steps
        lat_steps (int): Number of latitude steps

    Returns:
        tuple: (vertices, triangles, points) for the generated mesh
    """

    # Extract semi-axes
    a, b, c = semi_axes

    # Ensure a > b > c for the ellipsoidal coordinates
    #if not (a > b > c > 0):
    #   raise ValueError("Semi-axes must satisfy a > b > c > 0")

    # Create a grid for ellipsoidal harmonics
    longitudes = np.linspace(0, 360, lon_steps)
    latitudes = np.linspace(-89, 89, lat_steps) # Avoid exact poles (-90, 90)

    # Create a meshgrid for all combinations
    lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

    # Flatten the grid
    lons = lon_grid.flatten()
    lats = lat_grid.flatten()

    # Convert to radians
    phi = np.radians(lons)  # Azimuthal angle (longitude)
    theta = np.radians(90 - lats)  # Polar angle (colatitude)

    # Calculate Cartesian coordinates on a unit sphere
    x_unit = np.sin(theta) * np.cos(phi)
    y_unit = np.sin(theta) * np.sin(phi)
    z_unit = np.cos(theta)

    # Calculate ellipsoidal coordinates (u, v, w)
    # We need to solve for u, v, w where:
    # x^2/(a^2-u) + y^2/(b^2-u) + z^2/(c^2-u) = 0 (and similar for v and w)
    # This is a simplification - in practice, this requires numerical methods

    # For each point, compute the ellipsoidal coordinates
    # Here we map spherical to ellipsoidal coordinates approximately
    # by using the squared coordinates and semi-axes ratios

    # Initialize combined harmonics array
    combined_eh = np.zeros_like(x_unit, dtype=float)

    # Add each harmonic with its weight
    for harmonic in harmonics:
        h2 = harmonic['h']**2  # h² parameter
        k2 = harmonic['k']**2  # k² parameter
        n = harmonic['n']  # Degree
        m = harmonic.get('m', 0)  # Order (default to 0 if not provided)
        weight = harmonic['weight']

        # Compute approximate ellipsoidal coordinates
        # This is a mapping from the unit sphere to the ellipsoidal coordinate system
        u = c**2 + (z_unit**2) * (a**2 - c**2)

        # Calculate ellipsoidal harmonic using scipy.special.ellip_harm
        # Note: ellip_harm takes (h, k, n, m, p) where p is related to our u, v coordinates
        p_values = u / a**2  # Normalized parameter

        eh_values = np.zeros_like(p_values)
        for i in range(len(p_values)):
            try:
                eh_values[i] = ellip_harm(h2, k2, n, m, p_values[i])
                if np.isnan(eh_values[i]) or np.isinf(eh_values[i]):
                    eh_values[i] = 0.0
            except (ValueError, OverflowError):
                # Handle potential numerical issues
                eh_values[i] = 0.0

        # Add to combined value with weight
        combined_eh += weight * eh_values

    # Add to combined value with weight
    combined_eh = np.nan_to_num(combined_eh, nan=0.0, posinf=1.0, neginf=-1.0)

    # Normalize for visualization (avoiding division by zero)
    eh_min = np.min(combined_eh)
    eh_max = np.max(combined_eh)
    if eh_min != eh_max:
        combined_normalized = (combined_eh - eh_min) / (eh_max - eh_min)
    else:
        combined_normalized = np.zeros_like(combined_eh)

    # Create base ellipsoid points
    base_x = a * x_unit
    base_y = b * y_unit
    base_z = c * z_unit
    base_distances = np.sqrt(base_x**2 + base_y**2 + base_z**2)

    # Create perturbed distances based on ellipsoidal harmonics
    distances = base_distances * (1.0 + amplitude * combined_normalized / base_distances.max())

    # Directly calculate perturbed coordinates. Ensure no division by zero or NaN values
    scale_factors = np.nan_to_num(distances / base_distances, nan=1.0, posinf=1.0, neginf=1.0)

    x_perturbed = base_x * scale_factors
    y_perturbed = base_y * scale_factors
    z_perturbed = base_z * scale_factors

    # Convert back to longitude/latitude for mesh generation
    r_perturbed = np.sqrt(x_perturbed**2 + y_perturbed**2 + z_perturbed**2)
    r_perturbed = np.maximum(r_perturbed, 1e-10)  # Avoid division by zero

    lat_perturbed = np.degrees(np.arcsin(np.clip(z_perturbed / r_perturbed, -1.0, 1.0)))
    lon_perturbed = np.degrees(np.arctan2(y_perturbed, x_perturbed)) % 360

    # Combine into points array
    points = np.column_stack((lon_perturbed, lat_perturbed, r_perturbed))

    # Final check to ensure no NaN values
    if np.isnan(points).any():
        # Replace any remaining NaNs with reasonable values
        points = np.nan_to_num(points, nan=0.0)
        print("Warning: NaN values were detected and replaced")

    # Generate mesh
    vertices, triangles = spherical_mesh(points)

    return vertices, triangles, points



In [ ]:
# Example usage of the improved ellipsoidal harmonics function
example_harmonics = [
    #{'h': 1, 'k': 1, 'n': 0, 'm': 0, 'weight': 1.0},  # Base shape
    {'h': 1, 'k': 1, 'n': 0, 'm': 1, 'weight': 0.4},  # First deformation
    {'h': 2, 'k': 2, 'n': 0, 'm': 2, 'weight': 0.8},  # Second deformation
]

vertices_example, triangles_example, points_example = create_ellipsoidal_harmonics_mesh(
    example_harmonics,
    semi_axes=(8, 8, 7), # X, Y, Z
    amplitude=4
)

# Generate a description of the harmonics used
example_desc = ", ".join([f"({h['h']},{h['k']},{h['n']},{h.get('m',0)}): {h['weight']}" for h in example_harmonics])
print(f"Generated mesh using ellipsoidal harmonics: {example_desc}")
print(f"Mesh created with {len(vertices_example)} vertices and {len(triangles_example)} triangles")


In [ ]:
# Visualize the example mesh
fig_example = visualize_mesh(vertices_example, triangles_example, "Ellipsoidal Harmonics", return_fig=True)
fig_example.update_layout(template='plotly_dark', showlegend=False)
fig_example.show(renderer='notebook')
